# 5.2 Regression

1. Load the Galton dataset into a Pandas DataFrame.
    * Dataset info: https://www.randomservices.org/random/data/Galton.html
    
2. Summarize the dataset:
    * Number of rows
    * Average height of male/female kids
    * Standard deviation of male/female kids
    
3. Create a training and test dataset. The test dataset should be at least 25%.

4. Create 2 regression models for predicting the child's height based on (i) father's height and (ii) mother's height.

5. Compute the model quality parameters: $R^{2}$ and $MSE$.

6. Create a multivariate regression model including both the mother's and father's height as features. How does $R^{2}$ change?


References:
* https://scikit-learn.org/stable/modules/linear_model.html
* https://scikit-learn.org/stable/model_selection.html
* <https://pygot.wordpress.com/2017/03/25/simple-linear-regression-with-galton/>


In [1]:
!pip install requests

In [2]:
%matplotlib inline
import csv
import requests # pip install requests for easy http request for CSV data
import numpy as np
import pandas as pd

In [3]:
df=pd.read_csv("https://ytliu0.github.io/Stat390EF-R-Independent-Study-archive/RMarkdownExercises/Galton.txt", sep="\t")

In [4]:
df.head()

,Family,Father,Mother,Gender,Height,Kids
0,1,78.5,67.0,M,73.2,4
1,1,78.5,67.0,F,69.2,4
2,1,78.5,67.0,F,69.0,4
3,1,78.5,67.0,F,69.0,4
4,2,75.5,66.5,M,73.5,4


In [5]:
num_rows = len(df)
print(f"Number of rows in the dataset: {num_rows}")

# 2. Average and Standard Deviation of height for male/female kids
# In the Galton dataset, 'Height' refers to the child's height and 'Gender' is 'M' or 'F'
summary_stats = df.groupby('Gender')['Height'].agg(['mean', 'std'])

print("\n--- Height Summary of Kids by Gender ---")
print("Average Height:")
print(summary_stats['mean'])
print("\nStandard Deviation:")
print(summary_stats['std'])

Number of rows in the dataset: 898

--- Height Summary of Kids by Gender ---
Average Height:
Gender
F    64.110162
M    69.228817
Name: mean, dtype: float64

Standard Deviation:
Gender
F    2.370320
M    2.631594
Name: std, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df[['Father', 'Mother']]
y = df['Height']

# Split the dataset: 75% for training, 25% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing data shape: X={X_test.shape}, y={y_test.shape}")

Training data shape: X=(673, 2), y=(673,)
Testing data shape: X=(225, 2), y=(225,)


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# --- Model 1: Father's Height Only ---
# We need to extract the column as a 2D array (DataFrame) for scikit-learn
X_train_father = X_train[['Father']]
X_test_father = X_test[['Father']]

# Train and predict
model_father = LinearRegression()
model_father.fit(X_train_father, y_train)
y_pred_father = model_father.predict(X_test_father)

# Compute metrics
r2_father = r2_score(y_test, y_pred_father)
mse_father = mean_squared_error(y_test, y_pred_father)

print("--- Model 1: Father's Height ---")
print(f"R^2 Score: {r2_father:.4f}")
print(f"MSE:       {mse_father:.4f}\n")

# --- Model 2: Mother's Height Only ---
X_train_mother = X_train[['Mother']]
X_test_mother = X_test[['Mother']]

# Train and predict
model_mother = LinearRegression()
model_mother.fit(X_train_mother, y_train)
y_pred_mother = model_mother.predict(X_test_mother)

# Compute metrics
r2_mother = r2_score(y_test, y_pred_mother)
mse_mother = mean_squared_error(y_test, y_pred_mother)

print("--- Model 2: Mother's Height ---")
print(f"R^2 Score: {r2_mother:.4f}")
print(f"MSE:       {mse_mother:.4f}")

--- Model 1: Father's Height ---
R^2 Score: 0.0413
MSE:       11.5653

--- Model 2: Mother's Height ---
R^2 Score: 0.0517
MSE:       11.4394


In [8]:
# --- Model 3: Both Father's and Mother's Height ---
# X_train and X_test already contain both columns

# Train and predict
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
y_pred_multi = model_multi.predict(X_test)

# Compute metrics
r2_multi = r2_score(y_test, y_pred_multi)
mse_multi = mean_squared_error(y_test, y_pred_multi)

print("--- Model 3: Multivariate (Father + Mother) ---")
print(f"R^2 Score: {r2_multi:.4f}")
print(f"MSE:       {mse_multi:.4f}\n")

--- Model 3: Multivariate (Father + Mother) ---
R^2 Score: 0.0764
MSE:       11.1419



Conclusion: How does R-squared change?

Increase in R-squared: As expected, the R-squared score increases to 0.0764 when combining both parents' heights in the multivariate model. Two features provide more information than one, which slightly lowers the MSE and improves the model's predictive power.

Overall Low R-squared: However, an R-squared of 0.0764 is still very low, meaning the model only explains about 7.6 percent of the variance in the child's height.

Why is it so low? This is primarily because we omitted a crucial feature: the child's gender. As we saw in our initial data summary, male and female children have significantly different average heights. Without knowing the child's gender, the parents' heights alone are not enough to make a precise prediction.